# Map and Environment

In [ ]:
import os
import pickle

import cartopy.crs as ccrs
import cmocean
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dotenv import find_dotenv, load_dotenv
from matplotlib import gridspec
from matplotlib.lines import Line2D

from nescafe23_fluctuations import tools

import seaborn as sns
import colorsys
_ = load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

### Poster presentation settings

In [ ]:
import matplotlib.font_manager as fm

# Add the font file
font_path = '/Users/john/Library/Fonts/RedHatDisplay-Medium.ttf'  # Update this path
fm.fontManager.addfont(font_path)

# Use the font
plt.rcParams['font.family'] = 'Red Hat Display'

In [ ]:
whoi_colors = ['#00B7BD', '#EE5340', '#0069b1', '#041e42']

In [ ]:
def lighten_color(hex_color, saturation_scale=0.5, lightness_scale=1.5):
    """
    Lighten a hex color similar to seaborn's tab20 palette
    
    Args:
        hex_color: Hex color string (e.g., '#1f77b4')
        saturation_scale: Multiply saturation by this (default 0.5)
        lightness_scale: Multiply lightness by this (default 1.5)
    """
    # Remove '#' if present
    hex_color = hex_color.lstrip('#')
    
    # Convert hex to RGB (0-1 range)
    r, g, b = tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))
    
    # Convert RGB to HLS
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    
    # Adjust saturation and lightness
    s_new = min(1.0, s * saturation_scale)
    l_new = min(1.0, l * lightness_scale)
    
    # Convert back to RGB
    r_new, g_new, b_new = colorsys.hls_to_rgb(h, l_new, s_new)
    
    # Convert to hex
    return '#{:02x}{:02x}{:02x}'.format(
        int(r_new * 255),
        int(g_new * 255),
        int(b_new * 255)
    )

In [ ]:
colors = [
    whoi_colors[0],
    lighten_color(whoi_colors[0], lightness_scale=2),
    whoi_colors[1],
    lighten_color(whoi_colors[1], lightness_scale=1.4, saturation_scale=1)
]

In [ ]:
colors

In [ ]:
plt.plot(np.arange(0,10), lw=2, c=colors[0])
plt.plot(np.arange(2,12), lw=2, c=colors[1])
plt.plot(np.arange(4,14), lw=2, c=colors[2])
plt.plot(np.arange(6,16), lw=2, c=colors[3])

### load environment data

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/map_bathy.nc"
bathy = xr.open_dataset(fn)["elevation"]

fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/FCTD_SoundSpeed_Colosi_v2.mat"
fctd = tools.get_fctd(fn=fn, r_smooth=3000)

fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/dt_global_twosat_phy_l4_20230430_vDT2024.nc"
ssh = xr.open_dataset(fn)

In [ ]:
dss = tools.open_receptions()

In [ ]:
ssh_slice = (
    ssh["sla"]
    .isel({"time": 0})
    .sel({"latitude": slice(38, 50), "longitude": slice(-70, -50)})
)

- Bathymetry data from GEBCO (will eventually be hi-res map from Brenden)
- FCTD is from Matthew Alford
- sea surface height is from copernicus, global ocean gridded sea surface height.

In [ ]:
plt.rcParams["text.usetex"] = False
plt.rcParams["font.size"] = 9

coords = {
    "SION": [39 + 37.194 / 60, -63 - 39.427 / 60],
    "SIOE": [38 + 45.906 / 60, -62 - 15.626 / 60],
}

fig = plt.figure(figsize=(3.3, 2.2))

ax1 = fig.add_axes(
    (0,0,1,1), projection=ccrs.Orthographic(central_longitude=-62, central_latitude=35)
)
ax1.set_extent([-65, -61, 37.5, 40], ccrs.Geodetic())
# ax1.coastlines()


## MAP
plt.sca(ax1)
# Add gridlines with labels only on bottom and left
gl = ax1.gridlines(
    draw_labels=True,
    dms=False,
    x_inline=False,
    y_inline=False,
    color="#dddddd",
    alpha=0.2,
)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True

# Add axis labels
ax1.set_xlabel("Longitude", transform=ax1.transAxes)
ax1.set_ylabel("Latitude", transform=ax1.transAxes)

# Bathymetry
bathy_plot = (-bathy).plot(
    cmap='mako',
    transform=ccrs.PlateCarree(),
    ax=ax1,
    vmax=6000,
    vmin=500,
    add_colorbar=False,
    rasterized=True,
)  # cbar_kwargs = {'label':'depth [m]'}, alpha=1, rasterized=True)

# cbar = plt.colorbar(bathy_plot, ax=ax1, boundaries=np.linspace(1000,5000,256), label='depth [m]')
# cbar.ax.invert_yaxis()
# cbar.ax.set_yticks(np.arange(1000,5001,1000))

bathy.plot.contour(
    x="lon",
    y="lat",
    transform=ccrs.PlateCarree(),
    ax=ax1,
    colors="k",
    linewidths=0.3,
    alpha=0.8,
    levels=np.linspace(-5000, -1000, 15),
    linestyles="-",
)

# FCTD
plt.scatter(
    x=fctd["longitude"],
    y=fctd["latitude"],
    c=whoi_colors[3],
    transform=ccrs.PlateCarree(),
    s=5,
    lw=0,
    zorder=100,
)
#plt.text(
#    -64-0.75,
#    39.25 - 0.2,
##    "Hydrographic\nSurvey",
#    transform=ccrs.PlateCarree(),
#    bbox=dict(facecolor="white", edgecolor="black", boxstyle="round"),
#)

# Sea Surface Height
ssh_slice.plot.contour(
    transform=ccrs.PlateCarree(), levels=[0.25], colors=colors[2], linewidths=2
)
plt.title("")

# Moorings
mooring_names = {
    'SION':'Acoustic\nReceiver',
    'SIOE':'Acoustic\nSource'
}
for mooring in coords.keys():
    ax1.scatter(
        coords[mooring][1],
        coords[mooring][0],
        transform=ccrs.PlateCarree(),
        c="w",
        edgecolor="k",
        lw=1,
        s=40,
        zorder=1000,
    )
    #ax1.text(
    #    coords[mooring][1] + 0.25,
    #    coords[mooring][0] + 0.1,
    #    mooring_names[mooring],
    #    bbox=dict(facecolor="white", edgecolor="black", boxstyle="round"),
    #    transform=ccrs.PlateCarree(),
    #)

# legend
#custom_lines = [Line2D([0], [0], color=colors[2], lw=2)]
#leg = ax1.legend(custom_lines, ["0.25 m SSH contour"], loc='lower left')
#frame = leg.get_frame()
#frame.set_alpha(1.0)  # or any transparency value 0–1

#plt.xlabel('Longitude [ $^\\circ$ ]')
#plt.ylabel('Latitude [ $^\\circ$ ]')

plt.tight_layout()

fig.savefig("../../figures/presentation_figures/Map.pdf", dpi=500)